# Analyzing and Predicting Annual Corn Prices

This notebook loads USDA NASS annual corn price data (1866-2025) and applies three baseline forecasting methods and two regression models - OLS linear time trend and AR(1) autoregressive - to explore and forecast real corn prices.

Click the badge below to open in Google Colab:

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chuckgrigsby0/agec-370/blob/main/notebooks/20_predictive_modeling_corn_prices.ipynb)

## Data Loading

Import libraries and load the corn price dataset. The CSV includes both nominal and real (inflation-adjusted) prices per bushel from USDA NASS.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load from local data directory; swap for the GitHub raw URL when running in Colab
base_url = "https://raw.githubusercontent.com/chuckgrigsby0/agec-370/main/data/"
corn_prices = pd.read_csv(base_url + 'corn_prices_bushels_nominal_real.csv')

In [ ]:
corn_prices = corn_prices.sort_values('year').reset_index(drop=True)  # ascending year order for time series

## Data Visualization

The plot below shows real (inflation-adjusted) corn prices over time. Real prices remove the effect of general inflation so that price changes reflect actual shifts in corn market value.

In [ ]:
sns.set_style('whitegrid')  # clean background with subtle gridlines; avoids busy chart decoration

### Annual Real Corn Price - Line Plot

Documentation: [Seaborn Line Plot](https://seaborn.pydata.org/generated/seaborn.lineplot.html)

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(14, 6))  # wide canvas suits a century-long time series

sns.lineplot(
    data=corn_prices,
    x='year',
    y='real_price',
    linewidth=1.0,
    color='darkblue',  # dark blue contrasts cleanly against the white grid background
    ax=ax
)

ax.set_title("Annual Real Corn Prices ($/BU)", fontsize=14, weight='bold')
ax.set_xlabel("Year", fontsize=12, weight='bold')
ax.set_ylabel("Real Price ($/BU)", fontsize=12, weight='bold')

# Tick every 3 years prevents overlapping labels on a dense x-axis spanning 100+ years
all_years = sorted(corn_prices['year'].unique())
ax.set_xticks(all_years[::3])
ax.set_xticklabels(all_years[::3])
ax.tick_params(axis='x', rotation=90)

plt.tight_layout()
plt.show()

## Predictive Modeling

Predictive modeling uses historical data to forecast future values. We start with three simple baseline methods - mean, last observed price, and moving average. The baselines are benchmarks for more sophisticated predictive models. 

## Train/Test Split

A train/test split is a standard evaluation method in predictive modeling: fit the model on the training data, then measure accuracy on the test data, which is a held-out portion the fitted model has never seen.

For time series data, the split must respect temporal order. Using future observations to train a model that predicts the past produces misleadingly optimistic results. Here we hold out the last 5 years (2020-2024) as the test set and train on all prior years.

In [ ]:
cutoff_year = 2019  # last year of training data; 2020-2024 becomes the test set
train = corn_prices[corn_prices['year'] <= cutoff_year].copy()
test  = corn_prices[corn_prices['year'] >  cutoff_year].copy()

print(f"Training set: {train['year'].min()}-{train['year'].max()} ({len(train)} observations)")
print(f"Test set:     {test['year'].min()}-{test['year'].max()}  ({len(test)} observations)")

### Baseline 1: Mean Price

For every future year, predict using the historical average price.

$$\hat{y}_{T+h|T} = \bar{y} = (y_1 + \cdots + y_T)/T$$

**Note:** `h` is the forecast horizon, meaning how many steps ahead you're predicting

In [ ]:
avg_price = train['real_price'].mean()  # computed on training data
print(f"Average Real Corn Price: ${avg_price:.2f} per bushel")

In [ ]:
# np.full fills an array with a constant; here the constant is avg_price for every forecast step
predicted_avg_train = np.full(len(train['real_price']), avg_price)
predicted_avg_test  = np.full(len(test['real_price']),  avg_price)

fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(
    train['year'], train['real_price'],
    color='darkblue', linewidth=1.3,
    label='Actual real corn price'
)
ax.plot(
    train['year'], predicted_avg_train,
    color='crimson', linestyle='--', linewidth=2.0,
    label=f'Mean price forecast (${avg_price:.2f}/bu)'
)
# Solid line in the test window signals out-of-sample performance
ax.plot(
    test['year'], predicted_avg_test,
    color='crimson', linestyle='-', linewidth=2.0
)

ax.set_title('Real Corn Prices: Mean Price Forecast', fontsize=14, weight='bold')
ax.set_xlabel('Year', fontsize=12, weight='bold')
ax.set_ylabel('Real Price ($/BU)', fontsize=12, weight='bold')

# Tick every 3 years prevents overlapping labels on a dense x-axis spanning 100+ years
all_years = sorted(corn_prices['year'].unique())
ax.set_xticks(all_years[::3])
ax.set_xticklabels(all_years[::3])
ax.tick_params(axis='x', rotation=90)

ax.legend()
ax.grid(axis='both', alpha=0.3, linestyle='-', linewidth=0.5)
plt.tight_layout()
plt.show()

### Baseline 2: Last Observed Price

Predict that next year's price equals the most recent observed price.

$$\hat{y}_{T+h|T} = y_{T}$$

In [ ]:
last_price = train[train['year'] == cutoff_year]['real_price'].values[0]  # last training observation
print(f"Last observed real price in {cutoff_year}: ${last_price:.2f} per bushel")

In [ ]:
predicted_last_price_test = np.full(len(test['real_price']), last_price)

fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(
    train['year'], train['real_price'],
    color='darkblue', linewidth=1.3,
    label='Actual real corn price'
)
ax.plot(
    test['year'], predicted_last_price_test,
    color='crimson', linestyle='-', linewidth=2.0,
    label=f'Last price forecast (${last_price:.2f}/bu)'
)

ax.set_title('Real Corn Prices: Last Observed Price Forecast', fontsize=14, weight='bold')
ax.set_xlabel('Year', fontsize=12, weight='bold')
ax.set_ylabel('Real Price ($/BU)', fontsize=12, weight='bold')

all_years = sorted(corn_prices['year'].unique())
ax.set_xticks(all_years[::3])
ax.set_xticklabels(all_years[::3])
ax.tick_params(axis='x', rotation=90)

ax.legend()
ax.grid(axis='both', alpha=0.3, linestyle='-', linewidth=0.5)
plt.tight_layout()
plt.show()

### Baseline 3: Moving Average

A moving average forecast predicts next period's price as the average of the previous `m` observed prices. Using a short window (m=3) is more responsive to recent price changes than the long-run mean, while still smoothing out year-to-year variation.

$$\hat{y}_{t+1|t} = \frac{y_t + y_{t-1} + y_{t-2}}{3}$$

In [ ]:
# rolling(3) computes the 3-year window average; shift(1) ensures the prediction for year t
# uses only years t-1, t-2, and t-3, so no future data leaks into the forecast
corn_prices['ma_3'] = corn_prices['real_price'].rolling(window=3).mean().shift(1)

cutoff_year = 2019
train = corn_prices[corn_prices['year'] <= cutoff_year].copy()
test  = corn_prices[corn_prices['year'] >  cutoff_year].copy()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(
    train['year'], train['real_price'],
    color='darkblue', linewidth=1.3,
    label='Actual real corn price'
)
ax.plot(
    test['year'], test['ma_3'],
    color='crimson', linestyle='-', linewidth=2.0,
    label='3-year moving average forecast'
)

ax.set_title('Real Corn Prices: 3-Year Moving Average Forecast', fontsize=14, weight='bold')
ax.set_xlabel('Year', fontsize=12, weight='bold')
ax.set_ylabel('Real Price ($/BU)', fontsize=12, weight='bold')

all_years = sorted(corn_prices['year'].unique())
ax.set_xticks(all_years[::3])
ax.set_xticklabels(all_years[::3])
ax.tick_params(axis='x', rotation=90)

ax.legend()
ax.grid(axis='both', alpha=0.3, linestyle='-', linewidth=0.5)
plt.tight_layout()
plt.show()

## Regression Modeling

We will now build predictive models using regression techniques.

- **OLS linear time trend** ($P_{t} = \beta_{0} + \beta_{1}t + \epsilon_{t}$): Does price drift up or down over time on average? This fits a straight line through the data using Ordinary Least Squares. 
- **AR(1) autoregressive model** ($P_{t} = \beta_{0} + \beta_{1}P_{t-1} + \epsilon_{t}$): Does this year's price depend on last year's? A beta near 1 indicates high year-to-year price persistence.

We use `statsmodels` for both, which reports coefficients, standard errors, and p-values.

In [ ]:
# Drop rows without a real_price (some early years may be missing CPI-deflated values)
corn_prices = corn_prices[['year', 'real_price']].dropna().sort_values('year').reset_index(drop=True).copy()

# Integer time index starting at 0; gives the OLS slope a clean interpretation:
# the estimated $/bu change for each one-year step forward from 1913
corn_prices['t'] = range(len(corn_prices))  # t=0 is 1913, t=1 is 1914, ...

corn_prices.head()

### Train/Test Split for Regression Models

In [ ]:
cutoff_year = 2019
train = corn_prices[corn_prices['year'] <= cutoff_year].copy()
test  = corn_prices[corn_prices['year'] >  cutoff_year].copy()

print(f"Training set: {train['year'].min()}-{train['year'].max()} ({len(train)} observations)")
print(f"Test set:     {test['year'].min()}-{test['year'].max()}  ({len(test)} observations)")

In [ ]:
import statsmodels.formula.api as smf

# The formula API automatically includes an intercept
# 't' is our integer time index: t=0 is 1913, t=1 is 1914, etc.
ols_result = smf.ols("real_price ~ t", data=train).fit()
print(ols_result.summary())

### Interpreting the OLS Output

Focus on the `coef` column in the results table:

- **`Intercept`**: predicted price when `t = 0`, i.e., the estimated starting price in 1913
- **`t`**: average annual change in real corn price - a negative coefficient means prices trend downward over time on average
- **`R-squared`**: the share of price variation explained by the time trend alone
- **`P>|t|`**: p-value; values below 0.05 indicate a statistically significant coefficient

### R-squared

**R-squared** ($R^{2}$) measures the proportion of the outcome's variance explained by the predictor variables. $R^{2}$ ranges from 0 to 1, where higher values indicate improved model fit.

$R^{2} = 1 - \frac{SSR}{SST}$

where: 

$SSR = \sum (y_i - \hat{y}_i)^{2} = \sum (\hat{e}_i)^{2}$

$SST = \sum (y_i - \bar{y})^{2}$

### AR(1) Autoregressive Model

In [ ]:
from statsmodels.tsa.ar_model import AutoReg

# AR(1): this year's price is modeled as a linear function of last year's price
# lags=1 specifies one lag; old_names=False uses the current statsmodels naming convention
ar1_result = AutoReg(train['real_price'], lags=1, old_names=False).fit()
print(ar1_result.summary())

### Interpreting the AR(1) Output

Focus on the `real_price.L1` coefficient (the "L" stands for lag):

- A value near **1.0** means prices are highly persistent - last year's price is a strong predictor of this year's, which is typical in commodity markets
- The `const` term is the baseline level when the lagged price equals zero (rarely meaningful on its own)
- Compare the AR(1) R-squared to the OLS R-squared below to see which model explains more price variation

In [ ]:
# statsmodels AutoReg does not expose R-squared directly, so we compute it manually
# R² = 1 - (sum of squared residuals / total sum of squares)
y_actual = ar1_result.fittedvalues + ar1_result.resid
ss_res = np.sum(ar1_result.resid ** 2)
ss_tot = np.sum((y_actual - y_actual.mean()) ** 2)
r2_ar1 = 1 - ss_res / ss_tot

print(f"OLS R²:   {ols_result.rsquared:.4f}")
print(f"AR(1) R²: {r2_ar1:.4f}")

## Model Comparison

We evaluate model accuracy using two complementary measures: Root Mean Squared Error (RMSE) and Mean Squared Error (MSE). We generate predictions from each model for the held-out 2020-2024 test period and compute these metrics to assess out-of-sample forecasting performance.

**Root Mean Squared Error (RMSE)** measures average prediction error in the original units of the data ($/bu). Lower RMSE indicates better forecast accuracy.

$RMSE_{test} = \sqrt {\frac{\sum(y_{i} - \hat{y}_{i})^{2}}{N}}$

**Mean Squared Error (MSE)** measures average squared prediction error. MSE is mathematically equivalent to RMSE but expressed in squared units.

$MSE = \frac{\sum (y_i - \hat{y}_i)^{2}}{N} = \frac{SSR}{N}$

In [ ]:
ols_train = smf.ols("real_price ~ t", data=train).fit()

# The formula API matches on column names, so passing the test DataFrame
# with the same 't' column automatically generates the correct predictions
test['ols_predicted'] = ols_train.predict(test)
test[['year', 'real_price', 'ols_predicted']]

In [ ]:
ar1_train = AutoReg(train['real_price'], lags=1, old_names=False).fit()

# Forecast out-of-sample using integer positions in the full series
# start=len(train) is the index of the first observation after training ends
# dynamic=True uses each prior predicted value as input, producing a true multi-step forecast
ar1_preds = ar1_train.predict(
    start=len(train),
    end=len(train) + len(test) - 1,
    dynamic=True
)
test['ar1_predicted'] = ar1_preds.values
test[['year', 'real_price', 'ols_predicted', 'ar1_predicted']]

In [ ]:
# Calculate RMSE (penalizes large errors more than small ones; square root returns units to $/bu)
rmse_ols = np.sqrt(np.mean((test['real_price'] - test['ols_predicted']) ** 2))
rmse_ar1 = np.sqrt(np.mean((test['real_price'] - test['ar1_predicted']) ** 2))

# Calculate MSE
mse_ols = np.mean((test['real_price'] - test['ols_predicted']) ** 2)
mse_ar1 = np.mean((test['real_price'] - test['ar1_predicted']) ** 2)

print(f"OLS trend RMSE: ${rmse_ols:.2f}/bu")
print(f"AR(1)     RMSE: ${rmse_ar1:.2f}/bu")

print(f"\nOLS trend MSE:  {mse_ols:.2f}")
print(f"AR(1)     MSE:  {mse_ar1:.2f}")

An RMSE of $3.03 means the model's predictions are off by an average of about $3/bu (in root-mean-squared terms). It's the standard deviation of the forecast errors. 

### Comparison of prediction accuracy across models

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

# Training period shown as a thin line; test period shown with thicker lines and markers
# so the out-of-sample comparison stands out visually
ax.plot(train['year'], train['real_price'],
        color='steelblue', linewidth=1.2, label='Training data (1913-2019)')

ax.plot(test['year'], test['real_price'],
        color='black', linewidth=2.5, marker='o', label='Actual (2020-2024)')

# OLS extrapolates the long-run downward trend; typically underpredicts when prices recover
ax.plot(test['year'], test['ols_predicted'],
        color='crimson', linestyle='--', linewidth=2, marker='s',
        label=f'OLS trend forecast (RMSE=${rmse_ols:.2f}/bu)')

# AR(1) rolls forward from the last observed training value using momentum
ax.plot(test['year'], test['ar1_predicted'],
        color='darkorange', linestyle=':', linewidth=2, marker='^',
        label=f'AR(1) forecast (RMSE=${rmse_ar1:.2f}/bu)')

# Vertical line marks the boundary between in-sample fit and out-of-sample forecast
ax.axvline(cutoff_year, color='gray', linestyle=':', linewidth=1.2,
           label=f'Train/test cutoff ({cutoff_year})')

ax.set_title('Real Corn Prices: OLS Trend vs. AR(1) Forecast (2020-2024 Holdout)',
             fontsize=13, weight='bold')
ax.set_xlabel('Year', fontsize=11, weight='bold')
ax.set_ylabel('Real Price (2024 $/BU)', fontsize=11, weight='bold')
ax.legend()
plt.tight_layout()
plt.show()